<a target="_blank" href="https://colab.research.google.com/github/XPOZpublic/xpoz-cookbooks/blob/main/gemini/social-intelligence-with-xpoz-gemini.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Social Intelligence with XPOZ MCP + Gemini

This cookbook shows how to use [XPOZ](https://xpoz.ai) — a social-intelligence MCP server with **1.5 B+ indexed posts** across Twitter, Instagram, Reddit & TikTok — together with **Gemini** for real-time social media analysis.

**What you’ll build:**
1. **NVIDIA Brand Intelligence** — Pull tweets, Reddit threads & Instagram posts via XPOZ MCP, have Gemini analyse sentiment and themes, then render a styled HTML report.
2. **Bitcoin Event Monitor** — Count bullish vs bearish social signals around Bitcoin, ask Gemini for a prediction brief, and produce a second HTML dashboard.

**Prerequisites — add these as Colab Secrets (🔑 icon in the sidebar):**
| Secret name | Where to get it |
|---|---|
| `XPOZ_API_KEY` | Free at [xpoz.ai](https://xpoz.ai) — 100 K results/month |
| `GOOGLE_API_KEY` | [aistudio.google.com](https://aistudio.google.com) |

## 1 · Setup

In [ ]:
!pip install -q "mcp>=1.8" google-genai httpx nest_asyncio

In [ ]:
from google import genai
import json, re, csv, io, httpx, nest_asyncio, os
from datetime import datetime, timedelta
from IPython.display import HTML, display
from google.colab import userdata
from mcp.client.streamable_http import streamable_http_client
from mcp import ClientSession

nest_asyncio.apply()

# ── API keys from Colab Secrets ──────────────────────────────────────────
XPOZ_API_KEY = userdata.get("XPOZ_API_KEY")
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

# ── Connect to XPOZ MCP ─────────────────────────────────────────────
http_client = httpx.AsyncClient(
    headers={"Authorization": f"Bearer {XPOZ_API_KEY}"}
)
await http_client.__aenter__()

transport = await streamable_http_client(
    url="https://mcp.xpoz.ai/mcp",
    http_client=http_client,
).__aenter__()
read_stream, write_stream, _ = transport

session = ClientSession(read_stream, write_stream)
await session.__aenter__()
await session.initialize()

# ── Gemini client ─────────────────────────────────────────────────
gemini = genai.Client(api_key=GOOGLE_API_KEY)

# ── Helpers ──────────────────────────────────────────────────────
def days_ago(n: int) -> str:
    return (datetime.now() - timedelta(days=n)).strftime("%Y-%m-%d")

def _parse_xpoz(text: str) -> dict:
    """Parse XPOZ MCP compact tabular response into a Python dict."""
    result = {"success": "success: true" in text, "data": {}}
    hdr = re.search(r'results\[\d+\]\{([^}]+)\}:', text)
    if hdr:
        fields = hdr.group(1).split(',')
        rows = []
        for line in text[hdr.end():].split('\n'):
            if not line.startswith('    '):
                if rows or (line.strip() and not line.startswith(' ')):
                    break
                continue
            try:
                vals = next(csv.reader(io.StringIO(line.strip())))
                row = {}
                for i, f in enumerate(fields):
                    if i < len(vals):
                        v = vals[i].strip()
                        if v in ('null', ''):
                            row[f] = None
                        else:
                            try: row[f] = int(v)
                            except ValueError:
                                try: row[f] = float(v)
                                except ValueError: row[f] = v
                rows.append(row)
            except Exception:
                pass
        result["data"]["results"] = rows
    for m in re.finditer(r'^\s{2}(\w+):\s+(.+)$', text, re.MULTILINE):
        k, v = m.group(1), m.group(2).strip()
        if k.startswith('results'): continue
        try: result["data"][k] = int(v)
        except ValueError:
            try: result["data"][k] = float(v)
            except ValueError: result["data"][k] = v
    return result

async def call_xpoz(tool_name: str, params: dict) -> dict:
    """Call an XPOZ MCP tool and return parsed response."""
    result = await session.call_tool(tool_name, params)
    raw = result.content[0].text
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return _parse_xpoz(raw)

# ── Verify connection ──────────────────────────────────────────────
tools = await session.list_tools()
print(f"Connected to XPOZ MCP — {len(tools.tools)} tools available")
print(f"Sample tools: {', '.join(t.name for t in tools.tools[:6])}")

---
## 2 · NVIDIA Brand Intelligence Report

Fetch recent posts about **NVIDIA** from Twitter, Reddit and Instagram via XPOZ MCP, then ask Gemini to produce a structured sentiment analysis.

In [ ]:
BRAND = "NVIDIA"

# ── Twitter ──────────────────────────────────────────────────────
count_r = await call_xpoz("countTweets", {
    "phrase": BRAND, "startDate": days_ago(7)
})
tweet_count = count_r["data"].get("count", 0)

tw_r = await call_xpoz("getTwitterPostsByKeywords", {
    "query": BRAND, "startDate": days_ago(7), "limit": 50,
    "fields": ["text", "authorUsername", "likeCount", "retweetCount", "createdAtDate"]
})
tweets = tw_r["data"].get("results", [])

# ── Reddit ───────────────────────────────────────────────────────
rd_r = await call_xpoz("getRedditPostsByKeywords", {
    "query": BRAND, "startDate": days_ago(7), "limit": 30,
    "fields": ["title", "selftext", "authorUsername", "score", "subredditName", "createdAtDate"]
})
reddit_posts = rd_r["data"].get("results", [])

# ── Instagram ────────────────────────────────────────────────────
ig_r = await call_xpoz("getInstagramPostsByKeywords", {
    "query": BRAND, "startDate": days_ago(7), "limit": 20,
    "fields": ["caption", "username", "likeCount", "commentCount", "createdAtDate"]
})
ig_posts = ig_r["data"].get("results", [])

print(f"Collected: {len(tweets)} tweets · {len(reddit_posts)} Reddit posts · {len(ig_posts)} Instagram posts")
print(f"Total tweet volume (7 days): {tweet_count:,}")

In [ ]:
# ── Build context for Gemini ─────────────────────────────────────────
tw_text = "\n".join(
    f"@{p.get('authorUsername','?')}: {p.get('text','')} "
    f"[likes:{p.get('likeCount',0)}, RTs:{p.get('retweetCount',0)}]"
    for p in tweets
)
rd_text = "\n".join(
    f"r/{p.get('subredditName','')} — {p.get('title','')}: "
    f"{(p.get('selftext') or '')[:200]} [score:{p.get('score',0)}]"
    for p in reddit_posts
)
ig_text = "\n".join(
    f"@{p.get('username','?')}: {(p.get('caption') or '')[:200]} "
    f"[likes:{p.get('likeCount',0)}, comments:{p.get('commentCount',0)}]"
    for p in ig_posts
)

prompt = f"""Analyse the social-media sentiment for {BRAND} based on these posts from the last 7 days.
Total tweet volume: {tweet_count:,}

=== Twitter ({len(tweets)} posts) ===
{tw_text}

=== Reddit ({len(reddit_posts)} posts) ===
{rd_text}

=== Instagram ({len(ig_posts)} posts) ===
{ig_text}

Return your analysis in EXACTLY this format (keep the headers as-is):

OVERALL SENTIMENT: <Bullish/Bearish/Neutral> (<confidence>%)

KEY THEMES:
1. <theme> — <sentiment> — <one-line explanation>
2. …
3. …
4. …
5. …

RISK SIGNALS:
- <risk>
- …

OPPORTUNITIES:
- <opportunity>
- …

TOP POSTS:
1. <platform> @<author>: <quote> — <why it matters>
2. …
3. …"""

response = gemini.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
)
brand_analysis = response.text
print(brand_analysis)

In [ ]:
# ── Generate HTML report ─────────────────────────────────────────────
def _esc(t):
    t = t.replace('&','&amp;').replace('<','&lt;').replace('>','&gt;')
    t = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', t)
    return t.replace('\n\n','<br><br>').replace('\n','<br>')

rows_html = ""
for t in sorted(tweets, key=lambda x: (x.get('likeCount') or 0), reverse=True)[:12]:
    rows_html += (
        f'<tr><td>@{t.get("authorUsername","?")}</td>'
        f'<td>{(t.get("text") or "")[:120]}</td>'
        f'<td class="r">{t.get("likeCount",0)}</td>'
        f'<td class="r">{t.get("retweetCount",0)}</td></tr>'
    )

total_engagement = (
    sum((t.get('likeCount') or 0) for t in tweets)
    + sum((p.get('score') or 0) for p in reddit_posts)
    + sum((p.get('likeCount') or 0) for p in ig_posts)
)

html = f"""<!DOCTYPE html>
<html lang="en"><head><meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>{BRAND} Brand Intelligence</title>
<style>
*{{margin:0;padding:0;box-sizing:border-box}}
body{{background:#0f172a;color:#e2e8f0;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;padding:2rem;max-width:1100px;margin:0 auto}}
h1{{font-size:2.2em;background:linear-gradient(135deg,#a855f7,#ec4899);-webkit-background-clip:text;-webkit-text-fill-color:transparent}}
h2{{color:#a855f7;margin:2rem 0 .8rem;font-size:1.15em;border-bottom:1px solid #1e293b;padding-bottom:.5rem}}
.hdr{{text-align:center;margin-bottom:2.5rem}}
.meta{{color:#64748b;font-size:.85em;margin-top:.5rem}}
.badge{{display:inline-block;background:#a855f720;color:#a855f7;padding:3px 12px;border-radius:20px;font-size:.82em;font-weight:600;margin:4px 2px}}
.grid{{display:grid;grid-template-columns:repeat(4,1fr);gap:14px;margin:1.2rem 0}}
.card{{background:#1e293b;border-radius:12px;padding:1.2rem;text-align:center}}
.card .v{{font-size:1.9em;font-weight:800;color:#a855f7}}
.card .l{{color:#64748b;font-size:.82em;margin-top:4px}}
.box{{background:#1e293b;border-radius:12px;padding:1.5rem;margin:.8rem 0;line-height:1.75;color:#cbd5e1}}
.box strong{{color:#e2e8f0}}
table{{width:100%;border-collapse:collapse;margin:.8rem 0}}
th,td{{padding:7px 10px;text-align:left;border-bottom:1px solid #1e293b;font-size:.85em}}
th{{color:#94a3b8;font-size:.75em;text-transform:uppercase}}
.r{{text-align:right}}
.ft{{text-align:center;color:#475569;font-size:.78em;margin-top:2.5rem;padding-top:1.2rem;border-top:1px solid #1e293b}}
.ft a{{color:#a855f7}}
</style></head><body>
<div class="hdr">
  <h1>{BRAND} — Brand Intelligence Report</h1>
  <div><span class="badge">XPOZ MCP</span> <span class="badge">Gemini</span> <span class="badge">Twitter</span> <span class="badge">Reddit</span> <span class="badge">Instagram</span></div>
  <div class="meta">Data via XPOZ MCP &middot; Analysis by Gemini &middot; {datetime.now().strftime('%Y-%m-%d %H:%M')}</div>
</div>
<div class="grid">
  <div class="card"><div class="v">{tweet_count:,}</div><div class="l">Tweets (7 d)</div></div>
  <div class="card"><div class="v">{len(tweets)}</div><div class="l">Twitter samples</div></div>
  <div class="card"><div class="v">{len(reddit_posts) + len(ig_posts)}</div><div class="l">Reddit + IG</div></div>
  <div class="card"><div class="v">{total_engagement:,}</div><div class="l">Total engagement</div></div>
</div>
<h2>Gemini Analysis</h2>
<div class="box">{_esc(brand_analysis)}</div>
<h2>Top Tweets by Engagement</h2>
<table><thead><tr><th>Author</th><th>Text</th><th class="r">Likes</th><th class="r">RTs</th></tr></thead>
<tbody>{rows_html}</tbody></table>
<div class="ft">Powered by <a href="https://xpoz.ai">XPOZ</a> Social Intelligence MCP &middot; Gemini</div>
</body></html>"""

with open("nvidia-brand-intelligence.html", "w") as f:
    f.write(html)

print("Saved: nvidia-brand-intelligence.html")
display(HTML(html))

---
## 3 · Bitcoin Social Sentiment Monitor

Count **bullish vs bearish** tweets about Bitcoin, fetch samples, and ask Gemini for a prediction-market-style signal report.

In [ ]:
EVENT = "Bitcoin"
Q_BULL = '"bitcoin" AND ("bullish" OR "moon" OR "buy" OR "breakout" OR "rally" OR "ATH")'
Q_BEAR = '"bitcoin" AND ("bearish" OR "crash" OR "sell" OR "dump" OR "bubble" OR "overvalued")'

# ── Volume counts ──────────────────────────────────────────────────
bull_c = await call_xpoz("countTweets", {"phrase": Q_BULL, "startDate": days_ago(3)})
bear_c = await call_xpoz("countTweets", {"phrase": Q_BEAR, "startDate": days_ago(3)})
bullish_count = bull_c["data"].get("count", 0)
bearish_count = bear_c["data"].get("count", 0)
total_btc = bullish_count + bearish_count

# ── Sample posts ───────────────────────────────────────────────────
bull_r = await call_xpoz("getTwitterPostsByKeywords", {
    "query": Q_BULL, "startDate": days_ago(3), "limit": 25,
    "fields": ["text", "authorUsername", "likeCount", "retweetCount"]
})
bear_r = await call_xpoz("getTwitterPostsByKeywords", {
    "query": Q_BEAR, "startDate": days_ago(3), "limit": 25,
    "fields": ["text", "authorUsername", "likeCount", "retweetCount"]
})
bull_posts = bull_r["data"].get("results", [])
bear_posts = bear_r["data"].get("results", [])

# ── Reddit for extra colour ──────────────────────────────────────────
btc_rd = await call_xpoz("getRedditPostsByKeywords", {
    "query": "bitcoin", "startDate": days_ago(3), "limit": 20,
    "fields": ["title", "selftext", "score", "subredditName"]
})
btc_reddit = btc_rd["data"].get("results", [])

print(f"Bitcoin sentiment (3 days): {bullish_count:,} bullish · {bearish_count:,} bearish")
print(f"Ratio: {bullish_count/max(bearish_count,1):.1f}x bullish")
print(f"Samples: {len(bull_posts)} bullish tweets, {len(bear_posts)} bearish tweets, {len(btc_reddit)} Reddit posts")

In [ ]:
bull_text = "\n".join(
    f"@{p.get('authorUsername','?')}: {p.get('text','')} [likes:{p.get('likeCount',0)}]"
    for p in bull_posts[:15]
)
bear_text = "\n".join(
    f"@{p.get('authorUsername','?')}: {p.get('text','')} [likes:{p.get('likeCount',0)}]"
    for p in bear_posts[:15]
)
rd_btc_text = "\n".join(
    f"r/{p.get('subredditName','')} — {p.get('title','')}: {(p.get('selftext') or '')[:150]} [score:{p.get('score',0)}]"
    for p in btc_reddit
)

prompt2 = f"""Analyse social sentiment around "{EVENT}" to produce a prediction-market-style signal.

Volume (3 days): {bullish_count:,} bullish vs {bearish_count:,} bearish tweets.

=== Bullish tweets ===
{bull_text}

=== Bearish tweets ===
{bear_text}

=== Reddit ===
{rd_btc_text}

Return your analysis in EXACTLY this format:

SIGNAL: <BULLISH/BEARISH/NEUTRAL> (confidence: <1-10>)

BULL CASE:
1. <argument>
2. …
3. …

BEAR CASE:
1. <argument>
2. …
3. …

KEY VOICES:
- <@user>: <why they matter>
- …

CATALYSTS:
- <recent event or news driving sentiment>
- …

RISK FLIP:
- <what could reverse the current sentiment>
- …"""

resp2 = gemini.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt2,
)
btc_analysis = resp2.text
print(btc_analysis)

In [ ]:
# ── Generate Bitcoin HTML report ─────────────────────────────────────
bull_pct = bullish_count / max(total_btc, 1) * 100
bear_pct = 100 - bull_pct

bull_rows = ""
for p in sorted(bull_posts, key=lambda x: (x.get('likeCount') or 0), reverse=True)[:8]:
    bull_rows += (
        f'<tr><td>@{p.get("authorUsername","?")}</td>'
        f'<td>{(p.get("text") or "")[:120]}</td>'
        f'<td class="r">{p.get("likeCount",0)}</td></tr>'
    )
bear_rows = ""
for p in sorted(bear_posts, key=lambda x: (x.get('likeCount') or 0), reverse=True)[:8]:
    bear_rows += (
        f'<tr><td>@{p.get("authorUsername","?")}</td>'
        f'<td>{(p.get("text") or "")[:120]}</td>'
        f'<td class="r">{p.get("likeCount",0)}</td></tr>'
    )

html2 = f"""<!DOCTYPE html>
<html lang="en"><head><meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Bitcoin Social Sentiment Monitor</title>
<style>
*{{margin:0;padding:0;box-sizing:border-box}}
body{{background:#0a0e1a;color:#e2e8f0;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;padding:2rem;max-width:1100px;margin:0 auto}}
h1{{font-size:2.2em;background:linear-gradient(135deg,#f59e0b,#ef4444);-webkit-background-clip:text;-webkit-text-fill-color:transparent}}
h2{{color:#f59e0b;margin:2rem 0 .8rem;font-size:1.15em;border-bottom:1px solid #1e293b;padding-bottom:.5rem}}
.hdr{{text-align:center;margin-bottom:2.5rem}}
.meta{{color:#64748b;font-size:.85em;margin-top:.5rem}}
.badge{{display:inline-block;padding:3px 12px;border-radius:20px;font-size:.82em;font-weight:600;margin:4px 2px}}
.b-green{{background:#22c55e20;color:#22c55e}}
.b-red{{background:#ef444420;color:#ef4444}}
.b-yellow{{background:#f59e0b20;color:#f59e0b}}
.grid{{display:grid;grid-template-columns:repeat(3,1fr);gap:14px;margin:1.2rem 0}}
.card{{background:#1e293b;border-radius:12px;padding:1.2rem;text-align:center}}
.card .v{{font-size:1.9em;font-weight:800}}
.card .l{{color:#64748b;font-size:.82em;margin-top:4px}}
.green{{color:#22c55e}} .red{{color:#ef4444}} .yellow{{color:#f59e0b}}
.bar-wrap{{background:#1e293b;border-radius:12px;padding:1.2rem;margin:1rem 0}}
.bar{{display:flex;height:36px;border-radius:8px;overflow:hidden;margin-top:.5rem}}
.bar-bull{{background:#22c55e;display:flex;align-items:center;justify-content:center;font-weight:700;font-size:.85em}}
.bar-bear{{background:#ef4444;display:flex;align-items:center;justify-content:center;font-weight:700;font-size:.85em}}
.bar-label{{display:flex;justify-content:space-between;font-size:.8em;color:#94a3b8;margin-top:6px}}
.box{{background:#1e293b;border-radius:12px;padding:1.5rem;margin:.8rem 0;line-height:1.75;color:#cbd5e1}}
.box strong{{color:#e2e8f0}}
table{{width:100%;border-collapse:collapse;margin:.8rem 0}}
th,td{{padding:7px 10px;text-align:left;border-bottom:1px solid #1e293b;font-size:.85em}}
th{{color:#94a3b8;font-size:.75em;text-transform:uppercase}}
.r{{text-align:right}}
.ft{{text-align:center;color:#475569;font-size:.78em;margin-top:2.5rem;padding-top:1.2rem;border-top:1px solid #1e293b}}
.ft a{{color:#f59e0b}}
.cols{{display:grid;grid-template-columns:1fr 1fr;gap:16px}}
</style></head><body>
<div class="hdr">
  <h1>Bitcoin — Social Sentiment Monitor</h1>
  <div><span class="badge b-yellow">XPOZ MCP</span> <span class="badge b-yellow">Gemini</span> <span class="badge b-green">Bullish: {bullish_count:,}</span> <span class="badge b-red">Bearish: {bearish_count:,}</span></div>
  <div class="meta">3-day window &middot; Data via XPOZ MCP &middot; Analysis by Gemini &middot; {datetime.now().strftime('%Y-%m-%d %H:%M')}</div>
</div>
<div class="grid">
  <div class="card"><div class="v green">{bullish_count:,}</div><div class="l">Bullish posts</div></div>
  <div class="card"><div class="v red">{bearish_count:,}</div><div class="l">Bearish posts</div></div>
  <div class="card"><div class="v yellow">{bullish_count/max(bearish_count,1):.1f}x</div><div class="l">Bull / Bear ratio</div></div>
</div>
<div class="bar-wrap">
  <strong>Sentiment Meter</strong>
  <div class="bar"><div class="bar-bull" style="width:{bull_pct:.0f}%">{bull_pct:.0f}%</div><div class="bar-bear" style="width:{bear_pct:.0f}%">{bear_pct:.0f}%</div></div>
  <div class="bar-label"><span>Bullish</span><span>Bearish</span></div>
</div>
<h2>Gemini Analysis</h2>
<div class="box">{_esc(btc_analysis)}</div>
<h2>Sample Posts</h2>
<div class="cols">
<div>
<h3 style="color:#22c55e;font-size:.95em;margin-bottom:.5rem">Bullish</h3>
<table><thead><tr><th>Author</th><th>Text</th><th class="r">Likes</th></tr></thead><tbody>{bull_rows}</tbody></table>
</div>
<div>
<h3 style="color:#ef4444;font-size:.95em;margin-bottom:.5rem">Bearish</h3>
<table><thead><tr><th>Author</th><th>Text</th><th class="r">Likes</th></tr></thead><tbody>{bear_rows}</tbody></table>
</div>
</div>
<div class="ft">Powered by <a href="https://xpoz.ai">XPOZ</a> Social Intelligence MCP &middot; Gemini</div>
</body></html>"""

with open("bitcoin-sentiment-monitor.html", "w") as f:
    f.write(html2)

print("Saved: bitcoin-sentiment-monitor.html")
display(HTML(html2))

---
## Cleanup

In [ ]:
await session.__aexit__(None, None, None)
await http_client.aclose()
print("Done — MCP session closed.")